In [ ]:
import json
import numpy as np
import networkx as nx
from collections import defaultdict, Counter
from typing import Dict, List, Tuple
import os
import pickle
from sklearn.feature_extraction.text import CountVectorizer

import warnings
warnings.filterwarnings('ignore')

# For embeddings and similarity computation
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    print("Required libraries imported successfully!")
except ImportError as e:
    print(f"Missing library: {e}")
    print("Please install with: pip install sentence-transformers scikit-learn networkx")

np.random.seed(42)

: 

In [ ]:
from scipy.sparse import find, csr_matrix
import matplotlib.pyplot as plt
import pandas as pd
from scipy.linalg import norm
from IPython.core.display import HTML

# des options permettent de limiter (ou non) le nombre de lignes/colonnes affichées
# par exemple :
# pd.set_option('display.max_rows', None)

# cette fonction permet d'afficher une "jolie" représentation du vecteur v
# ARGS :
#   v : le vecteur à afficher (par ex. une ligne de la matrice X)
#   features : le vocabulaire
#   top_n : le nombre de mots maximum à afficher
def print_feats(v, features, top_n = 30):
    _, ids, values = find(v)
    feats = [(ids[i], values[i], features[ids[i]]) for i in range(len(list(ids)))]
    top_feats = sorted(feats, key=lambda x: x[1], reverse=True)[0:top_n]
    return pd.DataFrame({"word" : [t[2] for t in top_feats], "value": [t[1] for t in top_feats]})   

# fonction qui permet d'afficher plusieurs tables pandas côte à côte (c'est cadeau)
def display_side_by_side(dfs:list, captions:list):
    """Display tables side by side to save vertical space
    Input:
        dfs: list of pandas.DataFrame
        captions: list of table captions
    """
    output = ""
    combined = dict(zip(captions, dfs))
    for caption, df in combined.items():
        output += df.style.set_table_attributes("style='display:inline'").set_caption(caption)._repr_html_()
        output += "&emsp;"
        #output += "\xa0\xa0\xa0"
    display(HTML(output))

In [ ]:
def load_corpus(file_path: str) -> Dict[str, Dict]:
    """
    Load corpus data from JSONL file.
    Returns dictionary mapping document IDs to document data.
    """
    corpus = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():  # éviter les lignes vides
                obj = json.loads(line)
                corpus[obj["_id"]] = obj
    return corpus


def load_queries(file_path: str) -> Dict[str, Dict]:
    """
    Load query data from JSONL file.
    Returns dictionary mapping query IDs to query data.
    """
    queries = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                queries[obj["_id"]] = obj
    return queries


def load_qrels(file_path: str):
    """
    Load relevance judgments (TSV).
    Returns dict: query_id -> {doc_id: score}
    """
    qrels = {}
    with open(file_path, "r", encoding="utf-8") as f:
        next(f)  # sauter la ligne d'entête

        for line in f:
            if not line.strip():
                continue

            qid, docid, score = line.strip().split('\t')
            score = int(score)

            if qid not in qrels:
                qrels[qid] = {}

            qrels[qid][docid] = score

    return qrels

In [ ]:
# Load the dataset
print("Loading dataset...")
corpus = load_corpus('corpus.jsonl')
queries = load_queries('queries.jsonl')
qrels_valid = load_qrels('valid.tsv')


print(f"Loaded {len(corpus)} documents in corpus")
print(f"Loaded {len(queries)} queries")
print(f"Loaded relevance for {len(qrels_valid)} queries (dataset)")

2. Exploration des données et premier encodage

2.1 Statistiques simples sur les données

In [ ]:
import random
from collections import Counter

print("Taille du corpus :", len(corpus))
print("Nombre de requêtes :", len(queries))
print("Nombre de requêtes avec jugements (valid) :", len(qrels_valid))

# Nombre total de paires requête / document
nb_pairs = sum(len(cands) for cands in qrels_valid.values())
print("Nombre total de paires requête-document :", nb_pairs)

# Distribution des scores (pertinent = 1, non pertinent = 0)
all_scores = [score for cand_scores in qrels_valid.values() for score in cand_scores.values()]
score_counts = Counter(all_scores)

nb_pos = score_counts.get(1, 0)
nb_neg = score_counts.get(0, 0)

print("Nombre de documents pertinents   :", nb_pos)
print("Nombre de documents non pertinents :", nb_neg)
print("Proportion de pertinents :", nb_pos / nb_pairs if nb_pairs > 0 else 0)

# Statistiques par requête : combien de positifs / négatifs en moyenne
nb_pos_par_q = [sum(1 for s in cand_scores.values() if s == 1) for cand_scores in qrels_valid.values()]
nb_neg_par_q = [sum(1 for s in cand_scores.values() if s == 0) for cand_scores in qrels_valid.values()]

print("\nMoyenne de documents pertinents par requête :", np.mean(nb_pos_par_q))
print("Moyenne de documents non pertinents par requête :", np.mean(nb_neg_par_q))
print("Min / Max pertinents par requête :", np.min(nb_pos_par_q), "/", np.max(nb_pos_par_q))

2.2 Exemple de requête + candidats positifs / négatifs

In [ ]:
# On prend une requête au hasard parmi celles qui ont des jugements
example_qid = random.choice(list(qrels_valid.keys()))
example_query = queries.get(example_qid, {})

print("=== Exemple de requête ===")
print("Query ID :", example_qid)
print("Titre de la requête :", example_query.get("title", "(pas de titre)"))
print("Texte de la requête :", example_query.get("text", "")[:300], "...\n")

# Séparation des candidats pertinents / non pertinents
cand_scores = qrels_valid[example_qid]
pos_cands = [docid for docid, s in cand_scores.items() if s == 1]
neg_cands = [docid for docid, s in cand_scores.items() if s == 0]

print(f"Nombre de candidats pertinents (score=1) : {len(pos_cands)}")
print(f"Nombre de candidats non pertinents (score=0) : {len(neg_cands)}\n")

print("=== Titres de quelques candidats positifs ===")
for docid in pos_cands[:5]:
    doc = corpus.get(docid, {})
    print(f"- {docid} :: {doc.get('title', '(pas de titre)')}")

print("\n=== Titres de quelques candidats négatifs ===")
for docid in neg_cands[:5]:
    doc = corpus.get(docid, {})
    print(f"- {docid} :: {doc.get('title', '(pas de titre)')}")

2.3 Premier encodage : matrice Documents × Termes (CountVectorizer)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Liste des IDs + titres du corpus
corpus_ids = list(corpus.keys())
corpus_titles = [corpus[doc_id].get("title", "") for doc_id in corpus_ids]

# Vectorisation (sac de mots simple sur les titres)
vectorizer = CountVectorizer()  # paramètres par défaut au début
X = vectorizer.fit_transform(corpus_titles)

print("Matrice Documents x Termes construite.")
print("Nombre de documents :", X.shape[0])
print("Taille du vocabulaire :", X.shape[1])

# Vocabulaire (liste de mots)
features = vectorizer.get_feature_names_out()

2.4 Inspection de quelques vecteurs avec print_feats

In [ ]:
# Un document au hasard
idx = np.random.randint(0, X.shape[0])
doc_id = corpus_ids[idx]
v = X[idx]

print("=== Exemple de vecteur de document ===")
print("Document ID :", doc_id)
print("Titre :", corpus[doc_id].get("title", ""))
print("\nMots non nuls les plus importants dans ce document :")
print(print_feats(v, features, top_n=20))

2.5 Distribution des mots les plus fréquents

In [ ]:
# Somme des occurrences par colonne (sur tous les documents)
word_counts = np.asarray(X.sum(axis=0)).ravel()  # shape = (vocab_size,)

# On réutilise print_feats avec un "vecteur global"
from scipy.sparse import csr_matrix
v_global = csr_matrix(word_counts)

top_words_df = print_feats(v_global, features, top_n=30)
top_words_df

Histogramme:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

top_n = 30  # par exemple

# On récupère les top_n mots les plus fréquents
sorted_idx = np.argsort(word_counts)[::-1][:top_n]
top_words = features[sorted_idx]
top_counts = word_counts[sorted_idx]

plt.figure(figsize=(12, 5))
plt.bar(top_words, top_counts)
plt.title(f"Top {top_n} mots les plus fréquents (titres)")
plt.xlabel("Mots")
plt.ylabel("Fréquence")
plt.xticks(rotation=90)  # rotation pour éviter le chevauchement
plt.tight_layout()
plt.show()

3.1 Comparer des documents avec la similarité cosinus

a) Cosinus entre deux documents du corpus

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def cosine_between_docs(i: int, j: int) -> float:
    """
    Calcule la similarité cosinus entre les documents i et j
    (indices de lignes dans la matrice X).
    """
    return float(cosine_similarity(X[i], X[j])[0, 0])

# Exemple : comparer deux documents aléatoires
i, j = np.random.randint(0, X.shape[0], size=2)

doc_i_id = corpus_ids[i]
doc_j_id = corpus_ids[j]

print("Doc i :", doc_i_id)
print("Titre i :", corpus[doc_i_id].get("title", ""))

print("\nDoc j :", doc_j_id)
print("Titre j :", corpus[doc_j_id].get("title", ""))

sim_ij = cosine_between_docs(i, j)
print(f"\nSimilarité cosinus entre i et j : {sim_ij:.4f}")

b) Comparer un document avec plusieurs autres

Par exemple : document i vs 5 autres.

In [ ]:
i = np.random.randint(0, X.shape[0])
doc_i_id = corpus_ids[i]
print("Document de référence :", doc_i_id)
print("Titre :", corpus[doc_i_id].get("title", ""), "\n")

for _ in range(5):
    j = np.random.randint(0, X.shape[0])
    if j == i:
        continue
    doc_j_id = corpus_ids[j]
    sim_ij = cosine_between_docs(i, j)
    print(f"[cos={sim_ij:.3f}] {doc_j_id} :: {corpus[doc_j_id].get('title', '')}")

c) Variante : utiliser titre + résumé

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

corpus_fulltexts = [
    (corpus[doc_id].get("title", "") + " " + corpus[doc_id].get("text", ""))
    for doc_id in corpus_ids
]

vectorizer_full = CountVectorizer()
X_full = vectorizer_full.fit_transform(corpus_fulltexts)

features_full = vectorizer_full.get_feature_names_out()

def cosine_between_docs_full(i: int, j: int) -> float:
    return float(cosine_similarity(X_full[i], X_full[j])[0, 0])

3.2 Premier moteur de recherche (bag-of-words + cosinus)

a) Fonction de recherche

In [ ]:
def search_bow(query_text: str, top_k: int = 10):
    """
    Petit moteur de recherche basé sur :
    - CountVectorizer (titres)
    - similarité cosinus
    Retourne les top_k meilleurs documents.
    """
    # 1. Encoder la requête dans le même espace que les docs
    q_vec = vectorizer.transform([query_text])   # shape (1, vocab_size)

    # 2. Calculer la similarité cosinus avec tous les documents
    sims = cosine_similarity(q_vec, X)[0]        # shape (n_docs,)

    # 3. Trier par similarité décroissante
    ranked_idx = np.argsort(-sims)[:top_k]

    results = []
    for idx in ranked_idx:
        doc_id = corpus_ids[idx]
        title = corpus[doc_id].get("title", "")
        score = sims[idx]
        results.append((doc_id, score, title))
    return results

b) Tester le moteur

In [ ]:
query = "sentiment analysis on financial tweets"
results = search_bow(query, top_k=10)

print("=== Requête :", query, "===\n")
for doc_id, score, title in results:
    print(f"[{score:.3f}] {doc_id} :: {title}")

3.3 Variantes à tester (optionnel mais conseillé)

a) Modifier les paramètres du CountVectorizer

In [ ]:
vectorizer_tuned = CountVectorizer(
    stop_words="english",     # supprimer les stopwords
    max_df=0.8,               # ignorer les mots trop fréquents
    min_df=5,                 # ignorer les mots très rares
    ngram_range=(1, 2)        # unigrammes + bigrammes
)

X_tuned = vectorizer_tuned.fit_transform(corpus_titles)
features_tuned = vectorizer_tuned.get_feature_names_out()

b) Passer à TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.8,
    min_df=5
)

X_tfidf = tfidf_vectorizer.fit_transform(corpus_titles)
features_tfidf = tfidf_vectorizer.get_feature_names_out()

def search_tfidf(query_text: str, top_k: int = 10):
    q_vec = tfidf_vectorizer.transform([query_text])
    sims = cosine_similarity(q_vec, X_tfidf)[0]
    ranked_idx = np.argsort(-sims)[:top_k]

    results = []
    for idx in ranked_idx:
        doc_id = corpus_ids[idx]
        title = corpus[doc_id].get("title", "")
        score = sims[idx]
        results.append((doc_id, score, title))
    return results

Test rapide:

In [ ]:
query = "sentiment analysis on financial tweets"
for doc_id, score, title in search_tfidf(query, top_k=10):
    print(f"[{score:.3f}] {doc_id} :: {title}")

4.1 Choix du modèle & construction des textes à encoder

On va utiliser le modèle recommandé dans l’énoncé :
all-MiniLM-L6-v2 (rapide et généralement très correct).

On va encoder titre + résumé pour chaque article.

In [ ]:
from sentence_transformers import SentenceTransformer

# Nom du modèle (téléchargé automatiquement la première fois)
MODEL_NAME = "all-MiniLM-L6-v2"

# On fixe un ordre stable des documents (important pour retrouver les bons IDs)
corpus_ids = list(corpus.keys())  # ou sorted(corpus.keys()) si tu préfères
len(corpus_ids)

Construisons les textes d’entrée du modèle :

In [ ]:
# Texte d'entrée pour chaque doc : titre + résumé
corpus_texts_dense = [
    (corpus[doc_id].get("title", "") + " " + corpus[doc_id].get("text", ""))
    for doc_id in corpus_ids
]

4.2 Construction des embeddings avec SentenceTransformer

In [ ]:
import numpy as np

print("Chargement du modèle SentenceTransformer...")
model = SentenceTransformer(MODEL_NAME)

print("Encodage du corpus ")
corpus_embeddings = model.encode(
    corpus_texts_dense,
    batch_size=64,              
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   
)

print("Embeddings du corpus construits.")
print("Shape :", corpus_embeddings.shape)  # (nb_docs, dim_embedding)